# Seen and Unseen — write the freeze drafts (Exp 3 and Exp 2)

Run once, on the same kind of Colab runtime as the confirmatory runs (A100, High-RAM), after the code for the freeze is on `main`. About 15–25 minutes, mostly the Exp 2 dry run.

| Part | What it writes | Why it needs Colab |
|---|---|---|
| Exp 3 | `exp3_freeze_manifest.json`: binds the five configurations, the analyzer, the code commit and every pilot seed, and carries the environment block (lockfile hash, exact package versions and their hash, hardware class) | The environment must be recorded on the runtime that runs the confirmation |
| Exp 2 | `exp2_freeze_manifest.json`: binds the config, the episode manifest, the selection (recertified under the current V rule), the support and the code, plus the environment block (the owner's lockfile, versions, hardware class) | The Kuka cache is on Google Drive |

Both drafts are **unsigned**, and the runners and analyzers refuse them until the paper lead and the statistics check have signed (commands at the end). Downloads `freeze_drafts.zip`.

In [ ]:
#@title 0. Configuration
CODE_REF = "main"           #@param {type:"string"}   # the commit the freeze tag will point to
HARDWARE_EXP3 = "A100"      #@param {type:"string"}
HARDWARE_EXP2 = "A100"      #@param {type:"string"}   # where the 110 Exp 2 runs will train
EXP2_DRIVE_DIR = "/content/drive/MyDrive/seen-and-unseen/exp2"  #@param {type:"string"}
DRYRUN_DATASETS = 300       #@param {type:"integer"}
RUN_DRYRUN = True           #@param {type:"boolean"}   # False on a redo: reuses the dry run already on Drive (same runs)
RUN_EXP3 = True             #@param {type:"boolean"}   # False when the Exp 3 draft is already in approvals/
import os, torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE (switch to a GPU runtime)")
from google.colab import drive
drive.mount("/content/drive")
DRAFTS = "/content/freeze_drafts"
os.makedirs(DRAFTS, exist_ok=True)
OUT2 = f"{EXP2_DRIVE_DIR}/registered_out_freeze"
os.makedirs(OUT2, exist_ok=True)
CACHE = f"{EXP2_DRIVE_DIR}/kuka"
print("Exp 2 cache shards on Drive:", len([f for f in os.listdir(CACHE) if f.endswith(".npz")]) if os.path.exists(CACHE) else "MISSING")

## Exp 3

In [ ]:
assert "HARDWARE_EXP3" in globals(), "Run the configuration cell at the top first (after any restart: Runtime -> Run all)"
#@title 1. Code at the freeze commit and the confirmation's packages
%cd /content
import os
if not os.path.exists("trajectory-lineage"):
    !git clone -q https://github.com/physicalreasoning/trajectory-lineage.git
%cd /content/trajectory-lineage
!git fetch -q && git checkout -q {CODE_REF} && git rev-parse HEAD
!pip install -q numpy h5py "transformers>=5.0" accelerate

In [ ]:
assert "HARDWARE_EXP3" in globals(), "Run the configuration cell at the top first (after any restart: Runtime -> Run all)"
#@title 2. Environment block and the unsigned Exp 3 manifest
%cd /content/trajectory-lineage
if RUN_EXP3:
    !python scripts/freeze.py environment --hardware-class "{HARDWARE_EXP3}" --out {DRAFTS}/exp3_environment.json
    !python scripts/freeze.py draft --environment {DRAFTS}/exp3_environment.json --out {DRAFTS}/exp3_freeze_manifest.json
    !python scripts/freeze.py check --manifest {DRAFTS}/exp3_freeze_manifest.json || echo "(expected: refused until both internal roles sign)"
else:
    print("Exp 3 skipped (its draft is already in approvals/)")
print("draft written")

## Exp 2

In [ ]:
assert "HARDWARE_EXP3" in globals(), "Run the configuration cell at the top first (after any restart: Runtime -> Run all)"
#@title 3. Owner's pipeline (for its lockfile) and the registered package from the repository
%cd /content
import os
if not os.path.exists("rwmo"):
    !git clone -q https://github.com/subhashjprasad/robot-world-model-optimism-research.git rwmo
!git -C /content/rwmo rev-parse --short HEAD
!pip install -q scipy scikit-learn pandas

In [ ]:
assert "HARDWARE_EXP3" in globals(), "Run the configuration cell at the top first (after any restart: Runtime -> Run all)"
#@title 4. Recertify on the Drive cache with the current rules, then the unsigned Exp 2 freeze
%cd /content/trajectory-lineage/exp2-kuka-matched
!PYTHONPATH=. python -m registered.manifest build --npz-dir {CACHE} --out {OUT2}/manifest.jsonl | tail -n 12
!PYTHONPATH=. python -m registered.design --manifest {OUT2}/manifest.jsonl --out {OUT2}/selection.json | tail -n 14
!PYTHONPATH=. python -m registered.weights --manifest {OUT2}/manifest.jsonl --out {OUT2}/support.json | tail -n 3
if RUN_DRYRUN or not os.path.exists(f"{OUT2}/dryrun.md"):
    !PYTHONPATH=. python -m registered.dryrun --manifest {OUT2}/manifest.jsonl --selection {OUT2}/selection.json --support {OUT2}/support.json --datasets {DRYRUN_DATASETS} --workers 2 --out {OUT2}/dryrun.json | tail -n 10
else:
    print("dry run reused from Drive (same 110 runs; only the hashes changed)")
!PYTHONPATH=. python -m registered.freeze create --manifest {OUT2}/manifest.jsonl --selection {OUT2}/selection.json --support {OUT2}/support.json --hardware-class "{HARDWARE_EXP2}" --lockfile /content/rwmo/uv.lock --out {OUT2}/exp2_freeze_manifest.json | head -n 20
!PYTHONPATH=. python -m registered.freeze verify --freeze {OUT2}/exp2_freeze_manifest.json --selection {OUT2}/selection.json --support {OUT2}/support.json --lockfile /content/rwmo/uv.lock

## Download

In [ ]:
assert "HARDWARE_EXP3" in globals(), "Run the configuration cell at the top first (after any restart: Runtime -> Run all)"
#@title 5. Package the drafts and the recertified Exp 2 files
!cp {OUT2}/exp2_freeze_manifest.json {OUT2}/selection.json {OUT2}/selection.certificate.json {OUT2}/selection.md {OUT2}/support.json {OUT2}/support.md {OUT2}/dryrun.md {OUT2}/manifest.jsonl {OUT2}/manifest.summary.json {DRAFTS}/
!cd {DRAFTS} && zip -q -j /content/freeze_drafts.zip * && ls -la /content/freeze_drafts.zip
from google.colab import files
files.download("/content/freeze_drafts.zip")

### Signing (later, by the two internal signers)

```bash
# Exp 3 (from a clone of trajectory-lineage at the same commit)
python scripts/freeze.py sign --manifest exp3_freeze_manifest.json --role paper_lead --name "Full Name" --date YYYY-MM-DD
python scripts/freeze.py sign --manifest exp3_freeze_manifest.json --role stats_check --name "Full Name" --date YYYY-MM-DD
python scripts/freeze.py check --manifest exp3_freeze_manifest.json      # ok for all five configurations

# Exp 2 (from trajectory-lineage/exp2-kuka-matched)
PYTHONPATH=. python -m registered.freeze sign --freeze exp2_freeze_manifest.json --role paper_lead --name "Full Name" --date YYYY-MM-DD
PYTHONPATH=. python -m registered.freeze sign --freeze exp2_freeze_manifest.json --role stats_check --name "Full Name" --date YYYY-MM-DD
```

Then commit the signed Exp 3 manifest under `approvals/` (the only path allowed to change after the code commit), tag the code commit `exp3-freeze-YYYYMMDD` / `exp2-freeze-YYYYMMDD`, and register. The confirmatory runs start after registration.